# ML Coding — Day 5: Quantization I (fundamentals)

**~1 hour, vectorized NumPy.** Day 1 of the quantization arc — the bedrock of int8/int4 LLM serving:
affine quant/dequant, calibration, per-channel, fake-quant with the straight-through estimator
(backward!), and 4-bit bit-packing. `quantize` / `dequantize` from Q1 are reused later. No per-element
loops.

**Q1** affine quant/dequant `[warmup]` · **Q2** calibration (choose qparams) `[medium]` · **Q3**
per-channel weight quant `[medium]` · **Q4** fake-quant + STE backward `[hard]` · **Q5** 4-bit
pack/unpack `[hard · trick]`.

---

## Q1 — Uniform affine quantize / dequantize  ·  `[warmup]`

**Where it's real:** the int8 representation under every quantized model — map floats to integers with
a `scale` and integer `zero_point`, store ints, reconstruct approximately.

Implement (vectorized):
- `quantize(x, scale, zero_point, qmin, qmax) -> q`  = `clip(round(x/scale) + zero_point, qmin, qmax)`
  as an integer array.
- `dequantize(q, scale, zero_point) -> x_hat`  = `(q − zero_point)·scale`.

In-range round-trip error is bounded by `scale/2`; out-of-range values **saturate** at `qmin`/`qmax`.

In [ ]:
import numpy as np


def quantize(x, scale, zero_point, qmin, qmax):
    raise NotImplementedError


def dequantize(q, scale, zero_point):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    rng = np.random.default_rng(0)
    x = rng.uniform(-1, 1, 50)
    scale, zp, qmin, qmax = 2 / 255, 128, 0, 255
    q = quantize(x, scale, zp, qmin, qmax)
    assert q.dtype.kind in "iu" and q.min() >= qmin and q.max() <= qmax
    xh = dequantize(q, scale, zp)
    assert np.max(np.abs(x - xh)) <= scale / 2 + 1e-9
    assert quantize(np.array([100.0]), scale, zp, qmin, qmax)[0] == qmax     # saturate high
    assert quantize(np.array([-100.0]), scale, zp, qmin, qmax)[0] == qmin    # saturate low
    print("Q1 OK")

_q1()

## Q2 — Calibration: choose scale & zero-point  ·  `[medium]`

**Where it's real:** post-training quantization picks qparams from observed ranges (min/max
calibration). **Asymmetric** uses the full `[min, max]`; **symmetric** centers at 0 (`zero_point = 0`),
which is standard for weights and lets you skip zero-point correction in the int matmul.

Implement `choose_qparams(x, qmin, qmax, symmetric) -> (scale, zero_point)`:
- include 0 in the range (`lo = min(x.min(), 0)`, `hi = max(x.max(), 0)`);
- **asymmetric:** `scale = (hi − lo)/(qmax − qmin)`, `zero_point = round(qmin − lo/scale)` (clipped);
- **symmetric:** `scale = max(|lo|,|hi|)/qmax`, `zero_point = 0`.

`quantize/dequantize` (Q1) with these params should round-trip `x` to within ~one step.

In [ ]:
def choose_qparams(x, qmin, qmax, symmetric):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(1)
    x = rng.standard_normal(300) * 3 + 1
    s, z = choose_qparams(x, 0, 255, symmetric=False)             # asymmetric uint8
    xh = dequantize(quantize(x, s, z, 0, 255), s, z)
    assert np.max(np.abs(x - xh)) <= s + 1e-6
    s2, z2 = choose_qparams(x, -127, 127, symmetric=True)         # symmetric int8
    assert z2 == 0
    xh2 = dequantize(quantize(x, s2, z2, -127, 127), s2, z2)
    assert np.max(np.abs(x - xh2)) <= s2 / 2 + 1e-6
    print("Q2 OK")

_q2()

## Q3 — Per-channel weight quantization  ·  `[medium]`

**Where it's real:** LLM weight quantization is **per output channel**, not per tensor — one big
outlier channel would otherwise inflate a single scale and crush every other channel's precision.

Implement (symmetric, one scale per slice along `axis`):
- `quantize_per_channel(W, axis, qmax) -> (q, scales)`: `scales[c] = max(|W_c|)/qmax` over the channel
  `c`; `q = clip(round(W/scales), -qmax, qmax)`. Keep `scales` broadcastable (`keepdims`).
- `dequantize_per_channel(q, scales) -> W_hat = q·scales`.

Per-channel error must beat a single per-tensor scale when channels have very different magnitudes.

In [ ]:
def quantize_per_channel(W, axis, qmax):
    raise NotImplementedError


def dequantize_per_channel(q, scales):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(2)
    W = rng.standard_normal((4, 8)) * np.array([[1.], [10.], [100.], [1000.]])   # wildly different rows
    q, sc = quantize_per_channel(W, axis=0, qmax=127)
    Wh = dequantize_per_channel(q, sc)
    assert q.shape == W.shape and sc.shape == (4, 1)
    assert np.all(np.abs(W - Wh) <= sc * 0.5 + 1e-9)
    s = np.max(np.abs(W)) / 127
    Wt = np.clip(np.round(W / s), -127, 127) * s                  # per-tensor baseline
    assert np.max(np.abs(W - Wh)) < np.max(np.abs(W - Wt))
    print("Q3 OK")

_q3()

## Q4 — Fake-quant + straight-through estimator (backward)  ·  `[hard]`

**Papers:** Bengio et al., *Estimating gradients through stochastic neurons* (STE), 2013
(arXiv:1308.3432); Jacob et al., *Quantization and Training... integer-only inference*, 2017
(arXiv:1712.05877). **Why it matters:** quantization-aware training simulates quantization in the
forward pass (`fake_quant = dequantize(quantize(x))`) but `round` has zero gradient a.e. — so the
**STE** defines the backward as the identity *inside* the clipping range and 0 outside. (It's a
convention, not the true derivative.)

Implement:
- `fake_quant(x, scale, zero_point, qmin, qmax) -> x_hat` (= dequantize(quantize(x))).
- `fake_quant_backward(g, x, scale, zero_point, qmin, qmax) -> dx`: `dx = g` where
  `qmin ≤ round(x/scale)+zero_point ≤ qmax`, else `0`. No loops.

In [ ]:
def fake_quant(x, scale, zero_point, qmin, qmax):
    raise NotImplementedError


def fake_quant_backward(g, x, scale, zero_point, qmin, qmax):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(3)
    scale, zp, qmin, qmax = 0.1, 0, -8, 7
    x = rng.uniform(-1.5, 1.5, 60)
    y = fake_quant(x, scale, zp, qmin, qmax)
    assert np.allclose(y, dequantize(quantize(x, scale, zp, qmin, qmax), scale, zp))
    g = rng.standard_normal(60)
    dx = fake_quant_backward(g, x, scale, zp, qmin, qmax)
    qf = np.round(x / scale) + zp
    inr = (qf >= qmin) & (qf <= qmax)
    assert (~inr).any()                                 # some values are clipped (STE blocks them)
    assert np.allclose(dx[inr], g[inr]) and np.all(dx[~inr] == 0)
    print("Q4 OK")

_q4()

## Q5 — 4-bit pack / unpack  ·  `[hard · tensor trick]`

**Where it's real:** 4-bit weight formats — QLoRA's NF4 (Dettmers et al. 2023, arXiv:2305.14314) and
llama.cpp's GGUF k-quants — store **two 4-bit values per byte** with a per-block scale. **The trick:**
block-wise symmetric quantize to signed 4-bit, then pack pairs of nibbles into `uint8` with shifts and
masks (and unpack by reversing it).

Implement (block size even; `n` a multiple of `block`):
- `pack4(x, block) -> (packed_uint8, scales)`: per block, `scale = max(|·|)/7`, `q = clip(round(x/scale),
  -8, 7)`; map to nibble `q+8 ∈ [0,15]`; pack consecutive pairs as `lo | (hi << 4)`. `packed` has
  `n/2` bytes, `scales` has `n/block` entries.
- `unpack4(packed, scales, block, n) -> x_hat`: split each byte into low/high nibbles, undo `+8`,
  multiply by the block scale.

Round-trip error per block ≤ `scale/2`. No per-element loops (use reshape + bit ops).

In [ ]:
def pack4(x, block):
    raise NotImplementedError


def unpack4(packed, scales, block, n):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    rng = np.random.default_rng(4)
    n, block = 64, 16
    x = rng.standard_normal(n)
    packed, scales = pack4(x, block)
    assert packed.dtype == np.uint8 and packed.size == n // 2 and scales.size == n // block
    xh = unpack4(packed, scales, block, n)
    assert xh.shape == (n,)
    err = np.abs(x - xh).reshape(-1, block)
    assert np.all(err.max(axis=1) <= scales.reshape(-1) * 0.5 + 1e-9)
    p2, s2 = pack4(np.zeros(n), block)                  # all-zero block handled (no div by 0)
    assert np.allclose(unpack4(p2, s2, block, n), 0.0)
    print("Q5 OK")

_q5()

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np


def ref_quantize(x, scale, zero_point, qmin, qmax):
    q = np.round(np.asarray(x, float) / scale) + zero_point
    return np.clip(q, qmin, qmax).astype(np.int32)


def ref_dequantize(q, scale, zero_point):
    return (np.asarray(q, float) - zero_point) * scale


def ref_choose_qparams(x, qmin, qmax, symmetric):
    x = np.asarray(x, float)
    lo = min(x.min(), 0.0); hi = max(x.max(), 0.0)
    if symmetric:
        maxabs = max(abs(lo), abs(hi))
        scale = maxabs / qmax if maxabs > 0 else 1.0
        return scale, 0
    scale = (hi - lo) / (qmax - qmin)
    scale = scale if scale > 0 else 1.0
    zp = int(round(qmin - lo / scale))
    zp = int(np.clip(zp, qmin, qmax))
    return scale, zp


def ref_quantize_per_channel(W, axis, qmax):
    W = np.asarray(W, float)
    red = tuple(i for i in range(W.ndim) if i != axis)
    maxabs = np.max(np.abs(W), axis=red, keepdims=True)
    scales = np.where(maxabs > 0, maxabs / qmax, 1.0)
    q = np.clip(np.round(W / scales), -qmax, qmax).astype(np.int32)
    return q, scales


def ref_dequantize_per_channel(q, scales):
    return np.asarray(q, float) * scales


def ref_fake_quant(x, scale, zero_point, qmin, qmax):
    return ref_dequantize(ref_quantize(x, scale, zero_point, qmin, qmax), scale, zero_point)


def ref_fake_quant_backward(g, x, scale, zero_point, qmin, qmax):
    qf = np.round(np.asarray(x, float) / scale) + zero_point
    inrange = (qf >= qmin) & (qf <= qmax)
    return np.asarray(g, float) * inrange                  # STE: identity in range, 0 outside


def ref_pack4(x, block):
    x = np.asarray(x, float)
    xb = x.reshape(-1, block)
    scales = np.abs(xb).max(axis=1) / 7.0
    scales = np.where(scales > 0, scales, 1.0)
    q = np.clip(np.round(xb / scales[:, None]), -8, 7).astype(np.int8)
    nib = (q + 8).astype(np.uint8).reshape(-1)             # 0..15
    packed = (nib[0::2] | (nib[1::2] << 4)).astype(np.uint8)
    return packed, scales


def ref_unpack4(packed, scales, block, n):
    packed = np.asarray(packed, np.uint8)
    nib = np.empty(n, dtype=np.uint8)
    nib[0::2] = packed & 0x0F
    nib[1::2] = (packed >> 4) & 0x0F
    q = nib.astype(np.int16) - 8
    return (q.reshape(-1, block).astype(float) * scales[:, None]).reshape(-1)


_x = np.linspace(-1, 1, 9)
assert np.allclose(ref_dequantize(ref_quantize(_x, 2 / 255, 128, 0, 255), 2 / 255, 128), _x, atol=2 / 255)
_s, _z = ref_choose_qparams(np.array([-2.0, 3.0]), 0, 255, False)
assert ref_quantize(np.array([3.0]), _s, _z, 0, 255)[0] <= 255
assert ref_fake_quant_backward(np.ones(3), np.array([0.0, 100.0, -100.0]), 0.1, 0, -8, 7).tolist() == [1.0, 0.0, 0.0]
_p, _sc = ref_pack4(np.arange(8.0), 4)
assert _p.dtype == np.uint8 and _p.size == 4
print("reference solutions OK")